In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor


In [3]:
X_train_tab = np.load("X_train_tab.npy")
X_val_tab   = np.load("X_val_tab.npy")
y_train     = np.load("y_train.npy")
y_val       = np.load("y_val.npy")
X_test_tab  = np.load("X_test_tab.npy")

test_df = pd.read_csv("test2(test(1)).csv")


In [4]:
class HouseTabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


In [5]:
train_dataset = HouseTabularDataset(X_train_tab, y_train)
val_dataset   = HouseTabularDataset(X_val_tab, y_val)
test_dataset  = HouseTabularDataset(X_test_tab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)


In [6]:
class TabularModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


In [7]:
pip install torch 

Note: you may need to restart the kernel to use updated packages.


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TabularModel(X_train_tab.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(10):
    model.train()
    train_losses = []

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        preds = model(X).squeeze()
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---------- Validation ----------
    model.eval()
    preds = []
    true = []

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            outputs = model(X).squeeze().cpu().numpy()
            preds.extend(outputs)
            true.extend(y.numpy())

    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.2f} | "
        f"Val RMSE: {rmse:.2f} | "
        f"R²: {r2:.3f}"
    )



Epoch 1 | Train Loss: 412543590964.97 | Val RMSE: 624707.10 | R²: -2.110
Epoch 2 | Train Loss: 328138153883.11 | Val RMSE: 496072.58 | R²: -0.961
Epoch 3 | Train Loss: 166521184104.67 | Val RMSE: 331939.22 | R²: 0.122
Epoch 4 | Train Loss: 90620599982.03 | Val RMSE: 288267.39 | R²: 0.338
Epoch 5 | Train Loss: 77636135310.50 | Val RMSE: 275167.24 | R²: 0.397
Epoch 6 | Train Loss: 71589760005.04 | Val RMSE: 264096.12 | R²: 0.444
Epoch 7 | Train Loss: 66616984823.17 | Val RMSE: 253972.94 | R²: 0.486
Epoch 8 | Train Loss: 62385627700.97 | Val RMSE: 245285.91 | R²: 0.521
Epoch 9 | Train Loss: 58858470498.36 | Val RMSE: 237855.78 | R²: 0.549
Epoch 10 | Train Loss: 55961116480.32 | Val RMSE: 231812.86 | R²: 0.572


In [9]:
xgb_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train_tab, y_train)

val_preds_xgb = xgb_model.predict(X_val_tab)

rmse_xgb = np.sqrt(mean_squared_error(y_val, val_preds_xgb))
r2_xgb = r2_score(y_val, val_preds_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R²:", r2_xgb)


XGBoost RMSE: 186130.92613534164
XGBoost R²: 0.7239213585853577


In [10]:
test_preds = xgb_model.predict(X_test_tab)

submission = pd.DataFrame({
    "id": test_df["id"],
    "predicted_price": test_preds
})

submission.to_csv("23321020_final.csv", index=False)
print("23321020_final.csv generated successfully")


23321020_final.csv generated successfully
